<a href="https://colab.research.google.com/github/AhnafTouseef/Shotcut-video-editing/blob/main/Shotcut_video_editing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***`Action section`***

---



In [ ]:
mlt_path = "/content/drive/Othercomputers/My Laptop/drive/jp.mlt"   #@param{type:'string'}
output_name = "Trickcal Chibi Go - All that Glitters is not Gold 』【Japanese Voice】.mp4"    #@param{type:'string'}
method = "melt" #@param ['melt', 'decode_encode', 'smart_mux']
install_shotcut = False #@param {type:"boolean"}

# ***`File navigation commands`***

---



Setup script for easier files and folder navigation. It wraps OS level commands in python abstractions with commonly used words

In [ ]:
import requests

# Fetch the raw text of whatever script you need
script_url = "https://raw.githubusercontent.com/AhnafTouseef/colab-file-navigator/main/colab_tools.py"
remote_code = requests.get(script_url).text

# Run it immediately into the cell's memory
exec(remote_code)

# ***`Installs Shotcut`***

---



This script installs Shotcut on Google Colab.

In [ ]:
if install_shotcut:
    import os
    import subprocess

    if exists('/content/Shotcut'):
        print(f"Already installed")
    else:
        shotcut_archive = "/content/drive/MyDrive/Shotcut/shotcut-linux-x86_64-26.6.25.txz"
        extract_path = "/content"

        subprocess.run(["tar", "-xf", shotcut_archive, "-C", extract_path],check=True)
        print("Extraction complete.")
        rename("/content/Shotcut.app", "Shotcut")
else:
  pass

# ***`Preprocessors`***

---



Helper functions for preprocessing

In [ ]:
import re
import os

def remove_transitions(input_mlt, output_mlt=None):
    """
    Removes everything from the first <transition ...> tag
    to the last </transition> tag (inclusive).

    Parameters
    ----------
    input_mlt : str
        Path to the input .mlt file.

    output_mlt : str | None
        Path to save the modified file.
        If None, overwrites the input file.

    Returns
    -------
    str
        Path to the modified .mlt file.
    """

    if output_mlt is None:
        output_mlt = input_mlt

    with open(input_mlt, "r", encoding="utf-8") as f:
        text = f.read()

    pattern = r"<transition\b.*?</transition>.*(?=<\/tractor>)"

    matches = list(re.finditer(r"<transition\b.*?</transition>", text, flags=re.DOTALL))

    if matches:
        start = matches[0].start()
        end = matches[-1].end()
        text = text[:start] + text[end:]

    with open(output_mlt, "w", encoding="utf-8") as f:
        f.write(text)

    return output_mlt





def make_resource_paths_relative(input_mlt, output_mlt=None):
    """
    Replaces absolute paths in <property name="resource">...</property>
    with just the filename.

    Example:
        c:/Users/lenovo/Videos/drive/XL poster.mp4
            ↓
        XL poster.mp4
    """

    if output_mlt is None:
        output_mlt = input_mlt

    with open(input_mlt, "r", encoding="utf-8") as f:
        text = f.read()

    def replace(match):
        path = match.group(1)

        # Handle both Windows and Linux separators
        filename = os.path.basename(path.replace("\\", "/"))

        return f'<property name="resource">{filename}</property>'

    text = re.sub(
        r'<property name="resource">(.*?)</property>',
        replace,
        text,
        flags=re.DOTALL
    )

    with open(output_mlt, "w", encoding="utf-8") as f:
        f.write(text)

    return output_mlt



# ***`Render Video. (Modern Mode)`***

---



This script renders a video using Shotcut with Legacy Mode output

In [ ]:
import os, subprocess, time, re
from IPython.display import clear_output
# 1. Import rich components for live terminal rendering
# from rich.live import Live
# from rich.panel import Panel
# from rich.progress import Progress, BarColumn, TextColumn, TimeRemainingColumn


melt_path = "/content/Shotcut/melt"
mlt_project = mlt_path
output_video = f"/content/{output_name}.mp4"

print("Preprocessing")
# Note: Ensure remove_transitions and make_resource_paths_relative are defined in your environment
remove_transitions(mlt_project)
make_resource_paths_relative(mlt_project)

print("Starting render...")

command_cpu   = ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}"]
command_gpu_1 = ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}", "vcodec=h264_nvenc"]
command_gpu_2 = ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}", "vcodec=hevc_nvenc"]


def get_mlt_info(mlt_file):
    with open(mlt_file, "r", encoding="utf-8") as f:
        data = f.read()

    fps = re.search(r'frame_rate_num="(\d+)"\s+frame_rate_den="(\d+)"', data)
    duration = re.search(r'<tractor.*?out="(\d+):(\d+):(\d+\.\d+)"', data)

    fps = int(fps.group(1)) / int(fps.group(2))

    h, m, s = int(duration.group(1)), int(duration.group(2)), float(duration.group(3))
    seconds = h * 3600 + m * 60 + s

    return int(seconds * fps), fps


def format_time(sec):
    sec = int(max(sec, 0))
    return f"{sec//3600:02}:{(sec%3600)//60:02}:{sec%60:02}"


def render_with_melt(command, mlt_project, output_video):

    total_frames, fps = get_mlt_info(mlt_project)

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    start = time.time()
    calibration_time = 2
    first_frame = None
    speed = None
    current_frame = 0
    eta = None

    # ANSI Escape Codes for Colors
    CYAN = "\033[96m"
    MAGENTA = "\033[95m"
    YELLOW = "\033[93m"
    GREEN = "\033[92m"
    RED = "\033[91m"
    WHITE = "\033[97m"
    GREY = "\033[90m"
    RESET = "\033[0m"

    for line in process.stdout:
        match = re.search(r'frame[:= ]+(\d+)', line.lower())
        if match:
            current_frame = int(match.group(1))

        elapsed = time.time() - start

        if elapsed >= calibration_time:
            speed = current_frame / elapsed

        if speed:
            remaining_frames = total_frames - current_frame
            eta = remaining_frames / speed
            progress = current_frame / total_frames

            clear_output(wait=True)

            print(f"🎬 {CYAN}MELT RENDER{RESET}")
            print(f'{MAGENTA}Project name:{RESET} {mlt_project[46:]}')
            print(f'{MAGENTA}Rendering   :{RESET} {output_video[30:]}')
            print()

            bar = int(progress * 50)

            # The progress bar is now green blocks and red empty blocks
            if progress < 1:
                print(
                    # f"[{GREEN}{'█'*bar}{RED}{'░'*(50-bar)}{RESET}]"
                    f"┣{RED}{'━'*bar}{GREY}╋{'━'*(50-bar)}{RESET}┫"
                    f" {YELLOW}{progress*100:5.1f}%{RESET}"
                )
            else:
                print(
                    # f"[{GREEN}{'█'*bar}{RED}{'░'*(50-bar)}{RESET}]"
                    f"{GREEN}┣{'━'*bar}┫{RESET}"
                    f" {YELLOW}{progress*100:5.1f}%{RESET}"
                )

            print()
            print(f"{WHITE}Frames{RESET} : {CYAN}{current_frame:,}{RESET} / {CYAN}{total_frames:,}{RESET}")
            print(f"{WHITE}Speed{RESET}  : {YELLOW}{speed:.1f} fps{RESET}")
            print(f"{WHITE}Elapsed{RESET}: {GREEN}{format_time(elapsed)}{RESET}")
            print(f"{WHITE}ETA{RESET}    : {WHITE}{format_time(eta)}{RESET}", end=' ')

    process.wait()
    # clear_output(wait=True)

    if process.returncode == 0:
        print("\r✅ Render complete")
        print(f"{GREEN}{output_video[9:]}")
    else:
        print("❌ Render failed")



Preprocessing
Starting render...


# ***`Render Video. (Legacy Mode)`***

---



This script renders a video using Shotcut with Legacy Mode output

In [ ]:
# import os
# import subprocess

# output_name = "Trickcal Chibi Go -  Half-World Glove 』【Japanese Voice】"


# # ==========================
# # Render using melt + Xvfb
# # ==========================

# melt_path = "/content/Shotcut/melt"
# mlt_project = "/content/drive/Othercomputers/My Laptop/drive/test.mlt"
# output_video = f"/content/drive/Othercomputers/My Laptop/drive/{output_name}.mp4"
# # output_video = f"/content/{output_name}.mp4"

# print("preprossesing")
# remove_transitions(mlt_project)
# make_resource_paths_relative(mlt_project)

# print("Starting render...")

# command_cpu =   ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}"]
# command_gpu_1 = ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}", "vcodec=h264_nvenc"]
# command_gpu_2 = ["xvfb-run", "-a", melt_path, "-verbose", mlt_project, "-consumer", f"avformat:{output_video}", "vcodec=hevc_nvenc"]

# subprocess.run(command_gpu_1, check=True)

# print("Render complete.")
# print(f"Output: {output_video}")

To solve the greening problem with rendered video, all i had to do was remove transition tags from .mlt file. now the video renders perfectly. No all is left is to try gpu rendering.

It's interesting to see that gpu acceleration actually works and all the dependencies ar there from the begining.

***performance stats from a test video***



```
Colab CPU	      libx264	                  2 min

Colab T4	      Hardware encoder	          21 sec

Colab T4	      Other hardware encoder	  23 sec

my PC CPU	      libx264	                  1 min 14 sec

my Intel iGPU	  Hardware encoder	          39 sec
```



### Render video (ffmpeg method)

# ***`Fast methods method`***


---




In [ ]:
# @title Decode-Encode method
import os
import re
import subprocess
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm


def resolve_mlt_path(path, mlt_file):
    base_dir = os.path.dirname(os.path.abspath(mlt_file))

    if len(path) > 2 and path[1] == ":":
        path = os.path.basename(path.replace("\\", "/"))
    elif not os.path.isabs(path):
        path = path.replace("\\", "/")

    if not os.path.isabs(path):
        path = os.path.join(base_dir, path)

    return path


def time_to_seconds(t):
    h, m, s = t.split(":")
    return int(h)*3600 + int(m)*60 + float(s)


def shotcut_to_ffmpeg(mlt_file, output_file):
    root = ET.parse(mlt_file).getroot()

    chains = {}

    for chain in root.iter("chain"):
        cid = chain.get("id")

        for prop in chain.findall("property"):
            if prop.get("name") == "resource" and prop.text:
                chains[cid] = resolve_mlt_path(prop.text, mlt_file)
                break


    clips = []

    playlist_ids = []

    for tractor in root.iter("tractor"):
        for track in tractor.findall("track"):
            pid = track.get("producer")
            if pid:
                playlist_ids.append(pid)


    for playlist_id in playlist_ids:
        playlist = root.find(f".//*[@id='{playlist_id}']")

        if playlist is None:
            continue

        for entry in playlist.findall("entry"):
            cid = entry.get("producer")

            if cid in chains:
                clips.append({
                    "file": chains[cid],
                    "in": entry.get("in", "00:00:00.000"),
                    "out": entry.get("out")
                })


    if not clips:
        raise RuntimeError("No clips found.")


    total_duration = sum(
        time_to_seconds(c["out"]) - time_to_seconds(c["in"])
        for c in clips
    )


    print("\n===== TIMELINE =====")

    for i, clip in enumerate(clips, 1):
        print(f"{i}. {os.path.basename(clip['file'])}")
        print(f"   {clip['in']} -> {clip['out']}")

    print("====================\n")


    cmd = [
        "ffmpeg",
        "-y",
    ]


    for clip in clips:
        cmd += [
            "-ss", clip["in"],
            "-to", clip["out"],
            "-i", clip["file"]
        ]


    filters = ""

    for i in range(len(clips)):
        filters += (
            f"[{i}:v:0]"
            f"scale=1920:1080,"
            f"fps=30"
            f"[v{i}];"
            f"[{i}:a:0]"
            f"aresample=48000[a{i}];"
        )

    filters += "".join(
        f"[v{i}][a{i}]"
        for i in range(len(clips))
    )

    filters += (
        f"concat=n={len(clips)}:v=1:a=1"
        "[v][a]"
    )


    cmd += [
        "-filter_complex",
        filters,

        "-map", "[v]",
        "-map", "[a]",

        "-c:v", "h264_nvenc",
        "-c:a", "aac",

        output_file
    ]


    print("Running:")
    # print(" ".join(cmd))


    process = subprocess.Popen(
        cmd,
        stderr=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        text=True,
        bufsize=1
    )


    progress = tqdm(
        total=total_duration,
        unit="sec",
        desc="Rendering",
        dynamic_ncols=True
    )


    last_time = 0

    ffmpeg_error = []

    for line in process.stderr:
        # print(line, end="\r")   # temporary debug

        ffmpeg_error.append(line)

        match = re.search(r"time=(\d+:\d+:\d+\.\d+)", line)

        if match:
            current = time_to_seconds(match.group(1))

            if current > last_time:
                progress.update(current - last_time)
                last_time = current


    process.wait()
    progress.close()


    if process.returncode != 0:
        raise RuntimeError("FFmpeg render failed")


    print("\nRender complete:", output_file)

In [ ]:
# @title Video Mux Method

import os
import shutil
import subprocess
import xml.etree.ElementTree as ET
import json
from collections import Counter
from fractions import Fraction


# ==========================
# PATH HANDLING
# ==========================

def resolve_mlt_path(path, mlt_file):
    base = os.path.dirname(os.path.abspath(mlt_file))

    # Shotcut Windows path
    if len(path) > 2 and path[1] == ":":
        path = os.path.basename(path.replace("\\", "/"))

    return os.path.join(base, path)


# ==========================
# FFPROBE METADATA
# ==========================

def get_metadata(video):
    cmd = ["ffprobe","-v", "quiet","-print_format", "json","-show_streams","-show_format",video]
    data = json.loads(subprocess.check_output(cmd))
    video_stream = next(s for s in data["streams"]if s["codec_type"] == "video")
    audio_stream = next((s for s in data["streams"]if s["codec_type"] == "audio"),None)

    return {
        "video":  {
                  "codec": video_stream.get("codec_name"),
                  "profile": video_stream.get("profile"),
                  "level": video_stream.get("level"),
                  "width": video_stream.get("width"),
                  "height": video_stream.get("height"),
                  "pix_fmt": video_stream.get("pix_fmt"),
                  "fps": video_stream.get("r_frame_rate"),
                  "avg_fps": video_stream.get("avg_frame_rate"),
                  "time_base": video_stream.get("time_base"),
                  "bit_rate": video_stream.get("bit_rate")
                  },

        "audio":  {
                  "codec": (audio_stream.get("codec_name")if audio_stream else None),
                  "sample_rate": (audio_stream.get("sample_rate")if audio_stream else None),
                  "channels": (audio_stream.get("channels")if audio_stream else None),
                  "channel_layout": (audio_stream.get("channel_layout")if audio_stream else None),
                  "bit_rate": (audio_stream.get("bit_rate")if audio_stream else None),
                  "time_base": (audio_stream.get("time_base")if audio_stream else None)
                  }
            }


# ==========================
# READ SHOTCUT MLT
# ==========================

def parse_mlt(mlt_file):
    root = ET.parse(mlt_file).getroot()
    chains = {}

    for chain in root.iter("chain"):
        cid = chain.get("id")

        for p in chain.findall("property"):
            if p.get("name") == "resource":
                chains[cid] = resolve_mlt_path(p.text,mlt_file)

    clips = []

    for tractor in root.iter("tractor"):
        for track in tractor.findall("track"):
            pid = track.get("producer")
            playlist = root.find(f".//*[@id='{pid}']")

            if playlist is None:
                continue

            for entry in playlist.findall("entry"):
                producer = entry.get("producer")

                if producer in chains:
                    clips.append({"file": chains[producer],"in": entry.get("in"),"out": entry.get("out")})

    return clips


# ==========================
# CREATE COMPARISON FINGERPRINT
# ==========================

def make_fingerprint(metadata):

    video = metadata["video"]
    audio = metadata["audio"]

    return (

        # VIDEO
        video["codec"],
        video["profile"],
        video["level"],
        video["width"],
        video["height"],
        video["pix_fmt"],
        video["fps"],
        video["avg_fps"],

        # AUDIO
        audio["codec"],
        audio["sample_rate"],
        audio["channels"],
        audio["channel_layout"]

    )


# ==========================
# NORMALIZE EXCEPTION
# ==========================

def reencode_clip(clip, target, out):

    video = target["video"]
    audio = target["audio"]

    cmd = [
        "ffmpeg",
        "-y",

        # Exact MLT section
        "-ss",
        clip["in"],

        "-to",
        clip["out"],

        "-i",
        clip["file"],

        # Explicitly select streams
        "-map",
        "0:v:0",

        "-map",
        "0:a:0?",

        # --------------------------
        # VIDEO
        # --------------------------

        "-vf",
        (
            f"scale="
            f"{video['width']}:"
            f"{video['height']}:"
            f"flags=lanczos,"
            f"fps={video['fps']}"
        ),

        "-pix_fmt",
        video["pix_fmt"],

        "-c:v",
        "h264_nvenc"
    ]


    # Match H.264 profile when available

    if video["profile"]:
        profile = video["profile"].lower()
        profile_map = {
            "baseline": "baseline",
            "main": "main",
            "high": "high"
        }

        if profile in profile_map:
            cmd += ["-profile:v",profile_map[profile]]


    # Match H.264 level when available

    if video["level"]:
        level = str(video["level"])

        if level.isdigit():
            level_value = (int(level) / 10)

            cmd += ["-level:v",str(level_value)]


    # Match source bitrate when available

    if video["bit_rate"]:
        cmd += ["-b:v",str(video["bit_rate"])]


    # --------------------------
    # AUDIO
    # --------------------------

    if audio["codec"]:
        # AAC is used for normalized clips
        # because your original method uses AAC.
        cmd += ["-c:a","aac"]

        if audio["sample_rate"]:
            cmd += ["-ar",str(audio["sample_rate"])]

        if audio["channels"]:
            cmd += ["-ac",str(audio["channels"]) ]

        if audio["bit_rate"]:
            cmd += ["-b:a",str(audio["bit_rate"])]


    # --------------------------
    # TIMESTAMP HANDLING
    # --------------------------

    cmd += [

        "-start_at_zero",

        "-avoid_negative_ts",
        "make_zero",

        # Generate a clean MP4
        "-movflags",
        "+faststart",

        out
    ]


    subprocess.run(cmd,check=True)


# ==========================
# MAIN PIPELINE
# ==========================

def smart_render(mlt_file, output):
    clips = parse_mlt(mlt_file)

    print("\n===== TIMELINE =====")

    for i, c in enumerate(clips,1):
        print( i, os.path.basename( c["file"] ), c["in"], "->", c["out"] )

    print("====================")


    # ==========================
    # SCAN METADATA
    # ==========================

    print(
        "\nScanning metadata..."
    )

    metas = []

    for c in clips:
        m = get_metadata(c["file"])

        metas.append(m)

        c["meta"] = m


    # ==========================
    # FIND MAJORITY FORMAT
    # ==========================

    fingerprints = [make_fingerprint(m)for m in metas]
    target_fp, count = Counter(fingerprints).most_common(1)[0]
    target_index = fingerprints.index(target_fp)
    target = metas[target_index]

    print("\nTarget format:")
    print(target)
    print("Used by:",count,"clips")


    # ==========================
    # NORMALIZED DIRECTORY
    # ==========================

    temp = "normalized"

    os.makedirs(temp,exist_ok=True)

    final = []


    # ==========================
    # NORMALIZE ONLY EXCEPTIONS
    # ==========================

    for i, c in enumerate(clips):
        fp = make_fingerprint(c["meta"])

        if fp == target_fp:
            # Already compatible
            final.append(c["file"])

        else:
            print("\nReencoding:",os.path.basename(c["file"]))
            out = os.path.join(temp,f"{i}.mp4")
            reencode_clip(c,target,out)
            final.append(out)


    # ==========================
    # CONCAT LIST
    # ==========================

    concat_file = "concat.txt"

    with open(concat_file,"w",encoding="utf-8") as f:
        for x in final:
            f.write(f"file '{os.path.abspath(x)}'\n")


    # ==========================
    # STREAM COPY
    # ==========================

    cmd = [

        "ffmpeg",
        "-y",

        "-f",
        "concat",

        "-safe",
        "0",

        "-i",
        concat_file,

        "-c",
        "copy",

        output
    ]


    subprocess.run(cmd,check=True)


    # ==========================
    # CLEANUP
    # ==========================

    os.remove(concat_file)
    shutil.rmtree(temp,ignore_errors=True)


    print("\nDONE:",output)

# ***`Monitor`***

---



In [ ]:
# @title
def fast_render(mlt_file, output_file):

    if method == "decode_encode":
        shotcut_to_ffmpeg(mlt_file, output_video)

    elif method == "smart_mux":
        smart_render(mlt_file,output_file)

if method == "melt":
    render_with_melt(command_gpu_1, mlt_project, output_video)
else:
    fast_render(mlt_project, output_video)

🎬 MELT RENDER
Project name: 
Rendering   : 

┣━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┫ 100.0%

Frames : 7,228 / 7,227
Speed  : 56.0 fps
Elapsed: 00:02:09
✅ Render complete
sumon.mp4
